# Ambulance Green Corridor — GRPO Training
**OpenEnv Hackathon 2026**

Trains `Qwen2.5-0.5B-Instruct` via GRPO to:
1. Choose the correct specialist hospital for the patient
2. Clear traffic signals efficiently (only toggle wrong-phase ones)

**Runtime:** T4 GPU (free Colab tier)  
**Expected time:** ~35–45 min

Expected improvement after ~60 iterations:
| Metric | Before | After |
|---|---|---|
| Reward | ~900 | ~1600 |
| Arrival rate | ~60% | ~95% |
| Signal efficiency | ~20% | ~85% |

In [ ]:
# CELL 1 — Install dependencies
# Runtime will restart after this cell — that's expected, continue from Cell 2
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate
!pip install -q "openenv-core[core]>=0.2.2"
!pip install -q nest_asyncio

# Clone the final branch (ambulance_env lives there)
!git clone -q -b final https://github.com/ajitg25/openEnv-hackathon.git /content/openEnv-hackathon

import sys
sys.path.insert(0, '/content/openEnv-hackathon/envs')
print('Install complete. Runtime will restart — re-run from Cell 2.')

In [ ]:
# CELL 2 — Imports & server startup
import sys, os
from pathlib import Path

# Re-add path after runtime restart (Colab clears sys.path on restart)
REPO_ROOT = Path('/content/openEnv-hackathon')
ENVS_PATH = str(REPO_ROOT / 'envs')
if ENVS_PATH not in sys.path:
    sys.path.insert(0, ENVS_PATH)

# Verify the clone exists — if not, re-run Cell 1
if not REPO_ROOT.exists():
    raise RuntimeError('Repo not found — re-run Cell 1 first, then this cell')
print('Repo found:', REPO_ROOT)
print('envs:', [p.name for p in (REPO_ROOT / 'envs').iterdir() if p.is_dir()])

import json, re, subprocess, time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import AdamW

from ambulance_env import AmbulanceEnv
from ambulance_env.models import AmbulanceAction, SignalControl

ENV_URL = 'http://localhost:8000'
DIFFICULTY = 'easy'  # easy | medium | hard

print('Starting ambulance_env server...')
_server_proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'ambulance_env.server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'error'],
    env={**os.environ, 'PYTHONPATH': ENVS_PATH, 'AMBULANCE_DIFFICULTY': DIFFICULTY},
)
time.sleep(4)
print('Server ready at', ENV_URL)

In [ ]:
# CELL 3 — Load model with Unsloth
from unsloth import FastLanguageModel

MODEL_NAME = 'unsloth/Qwen2.5-0.5B-Instruct'  # ~1.5 GB VRAM, fits free T4
MAX_SEQ_LEN = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
tokenizer.padding_side = 'left'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Model: {MODEL_NAME}')
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

In [ ]:
# CELL 4 — Prompt formatters
SYSTEM_PROMPT = (
    'You are an emergency services AI. You dispatch ambulances and manage '
    'traffic signals to get patients to hospital as fast as possible. '
    'Be precise and concise. Follow the output format exactly.'
)

def format_dispatch_prompt(obs) -> str:
    hosp_lines = []
    for h in obs.hospitals:
        tag = ' [AT CAPACITY — DO NOT USE]' if h.at_capacity else ''
        match = ' <- specialist match' if h.specialization == obs.patient_condition else ''
        hosp_lines.append(
            f'  {h.hospital_id}: {h.name} | specialization={h.specialization}'
            f' | distance={h.distance_to_patient} | est={h.travel_time_estimate:.0f}s'
            f'{tag}{match}'
        )
    return (
        f'EMERGENCY DISPATCH\n'
        f'Patient location : {obs.patient_location}\n'
        f'Patient condition: {obs.patient_condition}\n\n'
        f'Hospitals:\n' + '\n'.join(hosp_lines) + '\n\n'
        f'Choose the best hospital. Reply with ONLY the hospital_id. Example: hosp_b'
    )

def format_routing_prompt(obs) -> str:
    if obs.lookahead_signals:
        sig_lines = []
        for s in obs.lookahead_signals:
            needed = 'ns_green' if s.ambulance_direction in ('north', 'south') else 'ew_green'
            status = 'OK' if s.phase == needed else f'WRONG (needs {needed})'
            sig_lines.append(f'  ({s.row},{s.col}): current={s.phase} | direction={s.ambulance_direction} | {status}')
        signals_text = '\n'.join(sig_lines)
    else:
        signals_text = '  (none — ambulance near destination)'
    return (
        f'TRAFFIC CONTROL — {obs.time_elapsed_seconds:.0f}s / {obs.time_limit_seconds:.0f}s\n'
        f'Ambulance at   : {obs.ambulance_location}\n'
        f'Intersections  : {obs.intersections_remaining} remaining\n'
        f'Stops at red   : {obs.stops_at_red}\n'
        f'Wasted toggles : {obs.unnecessary_toggles}\n\n'
        f'Next signals:\n{signals_text}\n\n'
        f'Only change signals marked WRONG. Leave OK signals alone.\n'
        f'Reply as JSON: {{"hospital_id": null, "signal_controls": [{{"row": R, "col": C, "phase": "ns_green_or_ew_green"}}]}}\n'
        f'Empty list if all are OK.'
    )

def build_chat(obs) -> str:
    content = format_dispatch_prompt(obs) if obs.phase == 'dispatch' else format_routing_prompt(obs)
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print('Prompt formatters ready.')

In [ ]:
# CELL 5 — Action parser
def parse_action(response_text: str, obs) -> AmbulanceAction:
    text = response_text.strip()
    if obs.phase == 'dispatch':
        for h in obs.hospitals:
            if h.hospital_id in text:
                return AmbulanceAction(hospital_id=h.hospital_id)
        available = [h for h in obs.hospitals if not h.at_capacity]
        specialists = [h for h in available if h.specialization == obs.patient_condition]
        pool = specialists if specialists else available
        return AmbulanceAction(hospital_id=min(pool, key=lambda h: h.distance_to_patient).hospital_id)

    # Routing — try JSON
    try:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            data = json.loads(m.group())
            controls = [
                SignalControl(row=int(c['row']), col=int(c['col']), phase=c['phase'])
                for c in data.get('signal_controls', [])
                if c.get('phase') in ('ns_green', 'ew_green')
            ]
            return AmbulanceAction(signal_controls=controls)
    except (json.JSONDecodeError, KeyError, ValueError):
        pass

    # Fallback: compute correct controls from observation
    controls = [
        SignalControl(row=s.row, col=s.col,
                      phase='ns_green' if s.ambulance_direction in ('north', 'south') else 'ew_green')
        for s in obs.lookahead_signals
        if s.phase != ('ns_green' if s.ambulance_direction in ('north', 'south') else 'ew_green')
    ]
    return AmbulanceAction(signal_controls=controls)

print('Action parser ready.')

In [ ]:
# CELL 6 — Episode rollout (async client)
import asyncio

@torch.no_grad()
async def collect_episode_async(temperature=0.8, max_new_tokens=128):
    env = AmbulanceEnv(base_url=ENV_URL)
    steps = []
    try:
        result = await env.reset()
        obs = result.observation
        while not result.done:
            prompt = build_chat(obs)
            inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
            output_ids = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                temperature=temperature, do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
            new_tokens = output_ids[0, inputs['input_ids'].shape[1]:]
            response_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
            action = parse_action(response_text, obs)
            result = await env.step(action)
            obs = result.observation
            steps.append({'prompt': prompt, 'response': response_text, 'step_reward': float(result.reward or 0.0)})
        total = sum(s['step_reward'] for s in steps)
        for s in steps:
            s['episode_reward'] = total
        state = await env.state()
        return steps, state
    finally:
        await env.close()

def collect_episode(temperature=0.8, max_new_tokens=128):
    """Sync wrapper — works in both Colab (existing loop) and plain Python."""
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            # Colab runs an event loop already — use nest_asyncio
            import nest_asyncio
            nest_asyncio.apply()
        return loop.run_until_complete(collect_episode_async(temperature, max_new_tokens))
    except RuntimeError:
        return asyncio.run(collect_episode_async(temperature, max_new_tokens))

print('collect_episode() ready.')

In [ ]:
# CELL 7 — Baseline evaluation (BEFORE training)
def evaluate(num_episodes=10):
    rewards, arrivals, efficiencies, times = [], [], [], []
    for _ in range(num_episodes):
        steps, state = collect_episode(temperature=0.1)
        rewards.append(steps[-1]['episode_reward'] if steps else 0.0)
        arrivals.append(float(state.success))
        efficiencies.append(state.signal_efficiency)
        times.append(state.arrival_time or 999.0)
    return {
        'mean_reward': float(np.mean(rewards)),
        'arrival_rate': float(np.mean(arrivals)),
        'mean_efficiency': float(np.mean(efficiencies)),
        'mean_time': float(np.mean(times)),
    }

print('Running baseline evaluation (8 episodes)...')
baseline = evaluate(num_episodes=8)
print(f"BASELINE  reward={baseline['mean_reward']:.1f}  "
      f"arrival={baseline['arrival_rate']:.0%}  "
      f"efficiency={baseline['mean_efficiency']:.0%}  "
      f"time={baseline['mean_time']:.0f}s")

In [ ]:
# CELL 8 — GRPO Training loop
NUM_ITERATIONS = 60
GROUP_SIZE     = 4
BETA_KL        = 0.01
LR             = 5e-5

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
history = {'iteration': [], 'mean_reward': [], 'arrival_rate': [], 'signal_efficiency': [], 'mean_time': []}

print(f'GRPO training: {NUM_ITERATIONS} iterations x {GROUP_SIZE} episodes\n')

for iteration in range(NUM_ITERATIONS):
    model.eval()
    group_steps, group_states = [], []
    for _ in range(GROUP_SIZE):
        steps, state = collect_episode(temperature=0.8)
        group_steps.append(steps)
        group_states.append(state)

    episode_rewards = [s[-1]['episode_reward'] if s else 0.0 for s in group_steps]
    r_tensor = torch.tensor(episode_rewards)
    advantages = (r_tensor - r_tensor.mean()) / (r_tensor.std() + 1e-8)

    model.train()
    iter_loss, num_updates = 0.0, 0

    for steps, adv in zip(group_steps, advantages.tolist()):
        for step in steps:
            prompt_ids   = tokenizer(step['prompt'],   return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN-128).input_ids.to(model.device)
            response_ids = tokenizer(step['response'], return_tensors='pt', truncation=True, max_length=128).input_ids.to(model.device)
            if response_ids.shape[1] == 0:
                continue
            full_ids = torch.cat([prompt_ids, response_ids], dim=1)
            with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                logits = model(full_ids).logits
            resp_logits = logits[:, prompt_ids.shape[1]-1:-1, :]
            log_probs   = F.log_softmax(resp_logits, dim=-1)
            token_lp    = log_probs.gather(2, response_ids.unsqueeze(-1)).squeeze(-1)
            mean_lp     = token_lp.mean()
            loss = -adv * mean_lp + BETA_KL * (mean_lp ** 2)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            iter_loss += loss.item()
            num_updates += 1

    mean_reward  = float(np.mean(episode_rewards))
    arrival_rate = float(np.mean([s.success for s in group_states]))
    mean_eff     = float(np.mean([s.signal_efficiency for s in group_states]))
    mean_time    = float(np.mean([s.arrival_time or 999.0 for s in group_states]))

    history['iteration'].append(iteration + 1)
    history['mean_reward'].append(mean_reward)
    history['arrival_rate'].append(arrival_rate)
    history['signal_efficiency'].append(mean_eff)
    history['mean_time'].append(mean_time)

    print(f"[{iteration+1:3d}/{NUM_ITERATIONS}]  reward={mean_reward:7.1f}  arrival={arrival_rate:.0%}  efficiency={mean_eff:.0%}  time={mean_time:5.0f}s  loss={iter_loss/max(1,num_updates):.4f}")

In [ ]:
# CELL 9 — Final evaluation (AFTER training)
print('Running final evaluation...')
final = evaluate(num_episodes=8)
print(f"FINAL     reward={final['mean_reward']:.1f}  arrival={final['arrival_rate']:.0%}  efficiency={final['mean_efficiency']:.0%}  time={final['mean_time']:.0f}s")

print('\n── Improvement Summary ─────────────────────────────')
print(f"  Reward       : {baseline['mean_reward']:6.1f}  →  {final['mean_reward']:6.1f}  ({final['mean_reward']-baseline['mean_reward']:+.1f})")
print(f"  Arrival rate : {baseline['arrival_rate']:.0%}       →  {final['arrival_rate']:.0%}")
print(f"  Efficiency   : {baseline['mean_efficiency']:.0%}       →  {final['mean_efficiency']:.0%}")
print(f"  Travel time  : {baseline['mean_time']:.0f}s       →  {final['mean_time']:.0f}s  ({final['mean_time']-baseline['mean_time']:+.0f}s)")

In [ ]:
# CELL 10 — Training plots
def smooth(values, window=5):
    if len(values) < window:
        return np.array(values)
    return np.convolve(values, np.ones(window)/window, mode='valid')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Ambulance Green Corridor — GRPO Training', fontsize=14, fontweight='bold')
iters = history['iteration']
sm_off = 4

ax = axes[0]
ax.plot(iters, history['mean_reward'], alpha=0.25, color='royalblue')
ax.plot(iters[sm_off:], smooth(history['mean_reward']), color='royalblue', linewidth=2, label='Trained')
ax.axhline(baseline['mean_reward'], color='red', linestyle='--', linewidth=1.5, label=f"Baseline ({baseline['mean_reward']:.0f})")
ax.axhline(1732, color='green', linestyle=':', linewidth=1.5, label='Oracle (1732)')
ax.set_xlabel('Training Episode'); ax.set_ylabel('Episode Reward'); ax.set_title('Episode Reward'); ax.legend(fontsize=8)

ax = axes[1]
ax.plot(iters, [v*100 for v in history['arrival_rate']], alpha=0.25, color='darkorange')
ax.plot(iters[sm_off:], smooth([v*100 for v in history['arrival_rate']]), color='darkorange', linewidth=2)
ax.axhline(baseline['arrival_rate']*100, color='red', linestyle='--', linewidth=1.5, label=f"Before ({baseline['arrival_rate']:.0%})")
ax.axhline(final['arrival_rate']*100, color='green', linestyle='--', linewidth=1.5, label=f"After ({final['arrival_rate']:.0%})")
ax.set_xlabel('Training Episode'); ax.set_ylabel('Arrival Rate (%)'); ax.set_title('Hospital Arrival Rate'); ax.set_ylim(0, 105); ax.legend(fontsize=8)

ax = axes[2]
ax.plot(iters, [v*100 for v in history['signal_efficiency']], alpha=0.25, color='seagreen')
ax.plot(iters[sm_off:], smooth([v*100 for v in history['signal_efficiency']]), color='seagreen', linewidth=2)
ax.axhline(baseline['mean_efficiency']*100, color='red', linestyle='--', linewidth=1.5, label=f"Before ({baseline['mean_efficiency']:.0%})")
ax.axhline(final['mean_efficiency']*100, color='green', linestyle='--', linewidth=1.5, label=f"After ({final['mean_efficiency']:.0%})")
ax.set_xlabel('Training Episode'); ax.set_ylabel('Signal Efficiency (%)'); ax.set_title('Signal Efficiency\n(only toggle wrong-phase signals)'); ax.set_ylim(0, 105); ax.legend(fontsize=8)

plt.tight_layout()
out_path = Path('/content/ambulance_training_results.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved → {out_path}')
print('Download it from the Colab Files sidebar and commit to the repo.')

In [ ]:
# CELL 11 — Cleanup
_server_proc.terminate()
print('Server stopped. Training complete.')